In [1]:
!rm -rf /root/.cache/huggingface/hub/models--microsoft--Phi-3-mini-4k-instruct

In [2]:
!pip install -U bitsandbytes>=0.46.1

In [3]:
import torch
import pandas as pd
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

In [4]:
model_id = "microsoft/Phi-3-mini-4k-instruct"
adapter_path = "./phi3-neuro-symbolic-adapter"


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False)
tokenizer.padding_side = "right"
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from transformers import BitsAndBytesConfig, AutoConfig, AutoModelForCausalLM
import torch

config = AutoConfig.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    trust_remote_code=False,
    force_download=True,
    revision="main"
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    config=config,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=False,
    attn_implementation="eager",
    revision="main"
)

In [7]:
import os
import zipfile
if not os.path.exists(adapter_path):
    if os.path.exists(adapter_path + ".zip"):
        print(f"Unzipping {adapter_path}.zip to {adapter_path}...")
        with zipfile.ZipFile(adapter_path + ".zip", 'r') as zip_ref:
            zip_ref.extractall(os.path.dirname(adapter_path))
    else:
        raise FileNotFoundError(f"Adapter path or zip not found: {adapter_path} or {adapter_path}.zip")

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

Unzipping ./phi3-neuro-symbolic-adapter.zip to ./phi3-neuro-symbolic-adapter...


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Phi3ForCausalLM(
      (model): Phi3Model(
        (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
        (layers): ModuleList(
          (0-31): 32 x Phi3DecoderLayer(
            (self_attn): Phi3Attention(
              (o_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
             

In [8]:
import pandas as pd
import torch
import time
import re

def extract_code(text):
    match = re.search(r'```python\n(.*?)\n```', text, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    return None

print("--- DEFINING THE EVALUATION PIPELINE (NO SANDBOX) ---")

def solve_problem(question, active_model):
    history = f"### Instruction: Write Python code to solve the math problem. You MUST store the final numerical answer in a variable exactly named 'result'.\n### Question:\n{question}\n### Code:\n"
    inputs = tokenizer(history, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = active_model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True
        )

    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    new_text = full_response[len(history):]
    code = extract_code(new_text)
    return code

def run_evaluation(test_df, active_model, model_name):
    print(f"\nRUNNING EVALUATION: {model_name}")
    print("Symbolic Sandbox & Agentic Loop: OFF")

    correct = 0
    total = len(test_df)

    for index, row in test_df.iterrows():
        target_answer = row['answer']
        agent_code = solve_problem(row['question'], active_model)
        if agent_code is not None and str(target_answer) in str(agent_code):
             correct += 1

    accuracy = (correct / total) * 100
    print(f"FINAL ACCURACY: {accuracy:.2f}% ({correct}/{total})")
    return accuracy
test_df = pd.read_csv("unified_svamp_test.csv").head(100)
run_acc = run_evaluation(test_df, model, "Fine-Tuned Model")

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


--- DEFINING THE EVALUATION PIPELINE (NO SANDBOX) ---

RUNNING EVALUATION: Fine-Tuned Model
Symbolic Sandbox & Agentic Loop: OFF


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


FINAL ACCURACY: 2.00% (2/100)
